# Homework 3

# Setup

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit, broadcast, hash, max, min
# from pyspark.sql.catalog import 

## Create Spark Session

In [2]:
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

25/07/18 19:00:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark

### Disable Automatic Broadcast Join

In [4]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

### Update system configurations to enable bucket join and preserve data grouping

In [5]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') 
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

## Build Spark Job

### Delete DDLs for each table for a fresh start

In [6]:
%%sql

DROP TABLE bootcamp.matches;

++
||
++
++

In [7]:
%%sql

DROP TABLE bootcamp.match_details;

++
||
++
++

In [8]:
%%sql

DROP TABLE bootcamp.medals_matches_players;

++
||
++
++

### First establish DDLs for each table.

In [9]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.medals_matches_players (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

++
||
++
++

In [10]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.match_details (
    match_id STRING, 
    player_gamertag STRING, 
    previous_spartan_rank STRING, 
    spartan_rank STRING, 
    previous_total_xp STRING, 
    total_xp STRING, 
    previous_csr_tier STRING, 
    previous_csr_designation STRING, 
    previous_csr STRING, 
    previous_csr_percent_to_next_tier STRING, 
    previous_csr_rank STRING, 
    current_csr_tier STRING, 
    current_csr_designation STRING, 
    current_csr STRING, 
    current_csr_percent_to_next_tier STRING, 
    current_csr_rank STRING, 
    player_rank_on_team STRING, 
    player_finished STRING, 
    player_average_life STRING, 
    player_total_kills STRING, 
    player_total_headshots STRING, 
    player_total_weapon_damage STRING, 
    player_total_shots_landed STRING, 
    player_total_melee_kills STRING, 
    player_total_melee_damage STRING, 
    player_total_assassinations STRING, 
    player_total_ground_pound_kills STRING, 
    player_total_shoulder_bash_kills STRING, 
    player_total_grenade_damage STRING, 
    player_total_power_weapon_damage STRING, 
    player_total_power_weapon_grabs STRING, 
    player_total_deaths STRING,
    player_total_assists STRING, 
    player_total_grenade_kills STRING, 
    did_win STRING, 
    team_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

25/07/18 19:00:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


++
||
++
++

In [11]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.matches (
    match_id STRING, 
    mapid STRING, 
    is_team_game STRING, 
    playlist_id STRING, 
    game_variant_id STRING, 
    is_match_over STRING, 
    completion_date STRING, 
    match_duration STRING, 
    game_mode STRING, 
    map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));

++
||
++
++

### Write CSV data to tables bucketing on match_id

In [12]:
matches_df = spark.read.option("header", "true").csv("/home/iceberg/data/matches.csv")
matches_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.matches")

In [13]:
match_details_df = spark.read.option("header", "true").csv("/home/iceberg/data/match_details.csv")
match_details_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.match_details")

In [14]:
medals_matches_players_df = spark.read.option("header", "true").csv("/home/iceberg/data/medals_matches_players.csv")
medals_matches_players_df.write \
  .format("iceberg") \
  .bucketBy(16, "match_id") \
  .mode("overwrite") \
  .saveAsTable("bootcamp.medals_matches_players")

### Read Data From Tables

In [15]:
matches_df = spark.table("bootcamp.matches")
matches_df

DataFrame[match_id: string, mapid: string, is_team_game: string, playlist_id: string, game_variant_id: string, is_match_over: string, completion_date: string, match_duration: string, game_mode: string, map_variant_id: string]

In [16]:
match_details_df = spark.table("bootcamp.match_details")
match_details_df

DataFrame[match_id: string, player_gamertag: string, previous_spartan_rank: string, spartan_rank: string, previous_total_xp: string, total_xp: string, previous_csr_tier: string, previous_csr_designation: string, previous_csr: string, previous_csr_percent_to_next_tier: string, previous_csr_rank: string, current_csr_tier: string, current_csr_designation: string, current_csr: string, current_csr_percent_to_next_tier: string, current_csr_rank: string, player_rank_on_team: string, player_finished: string, player_average_life: string, player_total_kills: string, player_total_headshots: string, player_total_weapon_damage: string, player_total_shots_landed: string, player_total_melee_kills: string, player_total_melee_damage: string, player_total_assassinations: string, player_total_ground_pound_kills: string, player_total_shoulder_bash_kills: string, player_total_grenade_damage: string, player_total_power_weapon_damage: string, player_total_power_weapon_grabs: string, player_total_deaths: stri

In [17]:
medal_matches_players_df = spark.table("bootcamp.medals_matches_players")
medal_matches_players_df

DataFrame[match_id: string, player_gamertag: string, medal_id: string, count: string]

### Bucket Join Medals Matches Players, Match Details, and Matches on 16 buckets

In [18]:
joined_df = medal_matches_players_df.join(match_details_df, on="match_id", how="inner")

joined_df = joined_df.join(matches_df, on="match_id")

joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#446, player_gamertag#447, medal_id#448, count#449, player_gamertag#375, previous_spartan_rank#376, spartan_rank#377, previous_total_xp#378, total_xp#379, previous_csr_tier#380, previous_csr_designation#381, previous_csr#382, previous_csr_percent_to_next_tier#383, previous_csr_rank#384, current_csr_tier#385, current_csr_designation#386, current_csr#387, current_csr_percent_to_next_tier#388, current_csr_rank#389, player_rank_on_team#390, player_finished#391, player_average_life#392, player_total_kills#393, player_total_headshots#394, ... 24 more fields]
   +- SortMergeJoin [match_id#446], [match_id#354], Inner
      :- Project [match_id#446, player_gamertag#447, medal_id#448, count#449, player_gamertag#375, previous_spartan_rank#376, spartan_rank#377, previous_total_xp#378, total_xp#379, previous_csr_tier#380, previous_csr_designation#381, previous_csr#382, previous_csr_percent_to_next_tier#383, previous_csr_ran

### Read medals and maps tables from CSV for broadcasting

In [19]:
medals_df = spark.read.option("header", "true").csv("/home/iceberg/data/medals.csv")
medals_df

DataFrame[medal_id: string, sprite_uri: string, sprite_left: string, sprite_top: string, sprite_sheet_width: string, sprite_sheet_height: string, sprite_width: string, sprite_height: string, classification: string, description: string, name: string, difficulty: string]

In [20]:
maps_df = spark.read.option("header","true").csv("/home/iceberg/data/maps.csv")
maps_df

DataFrame[mapid: string, name: string, description: string]

### Explicitly broadcast JOINs medals and maps

In [21]:
joined_df = joined_df.join(broadcast(medals_df), on="medal_id", how="inner")

joined_df = joined_df.join(broadcast(maps_df), on="mapid", how="inner")

joined_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [mapid#355, medal_id#448, match_id#446, player_gamertag#447, count#449, player_gamertag#375, previous_spartan_rank#376, spartan_rank#377, previous_total_xp#378, total_xp#379, previous_csr_tier#380, previous_csr_designation#381, previous_csr#382, previous_csr_percent_to_next_tier#383, previous_csr_rank#384, current_csr_tier#385, current_csr_designation#386, current_csr#387, current_csr_percent_to_next_tier#388, current_csr_rank#389, player_rank_on_team#390, player_finished#391, player_average_life#392, player_total_kills#393, ... 37 more fields]
   +- BroadcastHashJoin [mapid#355], [mapid#667], Inner, BuildRight, false
      :- Project [medal_id#448, match_id#446, player_gamertag#447, count#449, player_gamertag#375, previous_spartan_rank#376, spartan_rank#377, previous_total_xp#378, total_xp#379, previous_csr_tier#380, previous_csr_designation#381, previous_csr#382, previous_csr_percent_to_next_tier#383, previous_csr_ran